# 01 — Demand Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
sku = pd.read_csv("../data/raw/sku_master.csv")

print("Demand shape:", demand.shape)
print("SKU master shape:", sku.shape)
display(demand.head())
display(sku.head())

In [ ]:
print("Missing values:")
display(demand.isna().sum())
print("Duplicate rows:", demand.duplicated().sum())
print("Negative demand rows:", (demand["demand_units"] < 0).sum())
print("Unique SKUs:", demand["sku_id"].nunique())
print("Date range:", demand["week_start"].min(), "to", demand["week_start"].max())

In [ ]:
profile = (
    demand.groupby(["sku_id","sku_name","category"])
    .agg(
        avg_weekly_demand=("demand_units","mean"),
        demand_std=("demand_units","std"),
        total_demand=("demand_units","sum"),
        zero_demand_weeks=("demand_units", lambda s: (s==0).sum())
    ).reset_index()
)
profile["cv"] = profile["demand_std"] / profile["avg_weekly_demand"].replace(0, np.nan)
profile = profile.merge(sku[["sku_id","unit_cost"]], on="sku_id")
profile["annual_dollar_usage"] = profile["avg_weekly_demand"]*52*profile["unit_cost"]

profile = profile.sort_values("annual_dollar_usage", ascending=False).reset_index(drop=True)
profile["spend_share"] = profile["annual_dollar_usage"]/profile["annual_dollar_usage"].sum()
profile["cumulative_spend_share"] = profile["spend_share"].cumsum()
profile["abc_class"] = np.select(
    [profile["cumulative_spend_share"] <= 0.80,
     profile["cumulative_spend_share"] <= 0.95],
    ["A","B"], default="C"
)
profile["xyz_class"] = np.select(
    [profile["cv"] <= 0.35, profile["cv"] <= 0.70],
    ["X","Y"], default="Z"
)

profile.to_csv("../data/processed/sku_demand_profile.csv", index=False)
display(profile)